# Implementacion Algoritmo - Backtracking

In [ ]:
import time
from itertools import product, permutations

In [ ]:
class Tarea:
    def __init__(self, id: str, tiempo: int, prioridad: int, dependencias: list[str]):
        self.id = id
        self.tiempo = tiempo
        self.prioridad = prioridad
        self.dependencias = dependencias

    def __repr__(self):
        return str(self.id)

In [ ]:
def backtracking_servidores(tareas: list[Tarea], num_servidores: int):
    mapa_tareas = {t.id: t for t in tareas}
    total_tareas = len(tareas)

    mejor_tiempo = float("inf")
    mejor_costo = float("inf")
    mejor_orden = None
    mejor_asignacion = None

    def resolver(completadas: set, fin_tarea: dict, fin_servidor: list, orden_actual: list, asignacion_actual: list):
        nonlocal mejor_tiempo, mejor_costo, mejor_orden, mejor_asignacion

        tiempo_actual = max(fin_servidor, default=0)
        if tiempo_actual > mejor_tiempo:
            return

        if len(completadas) == total_tareas:
            costo_prioridad = sum(t.prioridad * fin_tarea[t.id] for t in tareas)

            if tiempo_actual < mejor_tiempo or (tiempo_actual == mejor_tiempo and costo_prioridad < mejor_costo):
                mejor_tiempo = tiempo_actual
                mejor_costo = costo_prioridad
                mejor_orden = list(orden_actual)
                mejor_asignacion = tuple(asignacion_actual)
            return

        disponibles = [
            t for t in tareas 
            if t.id not in completadas and all(dep in completadas for dep in t.dependencias)
        ]

        for tarea in disponibles:
            fin_deps = max((fin_tarea[dep] for dep in tarea.dependencias), default=0)

            for s in range(num_servidores):
                inicio = max(fin_servidor[s], fin_deps)
                fin = inicio + tarea.tiempo

                if fin > mejor_tiempo:
                    continue

                t_libre_previo = fin_servidor[s]
                fin_servidor[s] = fin
                fin_tarea[tarea.id] = fin
                completadas.add(tarea.id)
                orden_actual.append(tarea)
                asignacion_actual.append(s)

                resolver(completadas, fin_tarea, fin_servidor, orden_actual, asignacion_actual)

                fin_servidor[s] = t_libre_previo
                del fin_tarea[tarea.id]
                completadas.remove(tarea.id)
                orden_actual.pop()
                asignacion_actual.pop()

    resolver(
        completadas=set(),
        fin_tarea={},
        fin_servidor=[0] * num_servidores,
        orden_actual=[],
        asignacion_actual=[]
    )

    return (mejor_tiempo, mejor_orden, mejor_asignacion)